# 01 — DuckDB Columnar Storage Intuition

**Goal:** Understand why columnar storage (Parquet) is faster than row storage (CSV) for analytical queries.

DuckDB is an in-process analytical database that can query both Parquet and CSV files directly.
When a query only touches a few columns out of many, columnar formats like Parquet can skip
reading the untouched columns entirely. This notebook builds a synthetic dataset that mirrors
our Marshall Fire zonal-statistics table and measures the performance difference.

In [ ]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

In [ ]:
# ---------------------------------------------------------------------------
# Create a synthetic zonal_stats table
#   4,800 parcels x 5 observation dates = 24,000 rows
#   Columns mirror what we will compute in later notebooks.
# ---------------------------------------------------------------------------

np.random.seed(42)

N_PARCELS = 4800
DATES = ["2021-11-15", "2022-01-15", "2022-06-15", "2023-06-15", "2024-06-15"]

parcel_ids = np.repeat(np.arange(1, N_PARCELS + 1), len(DATES))
obs_dates = np.tile(DATES, N_PARCELS)

df = pd.DataFrame({
    "parcel_id": parcel_ids,
    "observation_date": obs_dates,
    "vv_change_db": np.random.normal(-2.0, 3.0, len(parcel_ids)),
    "vh_change_db": np.random.normal(-1.5, 2.8, len(parcel_ids)),
    "dnbr": np.random.normal(0.15, 0.25, len(parcel_ids)),
    "swir_b7": np.random.uniform(0.05, 0.45, len(parcel_ids)),
    "ndvi": np.random.normal(0.35, 0.2, len(parcel_ids)),
    "parcel_area_m2": np.random.uniform(200, 5000, len(parcel_ids)),
    "officially_destroyed": np.random.choice([0, 1], size=len(parcel_ids), p=[0.7, 0.3]),
})

out_dir = Path("../data/interim")
out_dir.mkdir(parents=True, exist_ok=True)

parquet_path = out_dir / "zonal_stats_synthetic.parquet"
csv_path = out_dir / "zonal_stats_synthetic.csv"

df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Parquet size: {parquet_path.stat().st_size / 1024:.1f} KB")
print(f"CSV size:     {csv_path.stat().st_size / 1024:.1f} KB")
df.head()

## Compare Parquet vs CSV query speed

We run the same analytical query — touching only 2 of the 9 columns — against
both formats. The query groups by `observation_date` and computes the average
`vv_change_db`. Parquet should be faster because DuckDB can skip reading the
other 7 columns entirely.

In [ ]:
QUERY_TEMPLATE = """
SELECT observation_date,
       AVG(vv_change_db) AS mean_vv_change,
       COUNT(*)          AS n
FROM   '{}'
GROUP  BY observation_date
ORDER  BY observation_date
"""

con = duckdb.connect()

# -- Parquet ---------------------------------------------------------------
t0 = time.perf_counter()
for _ in range(50):
    result_pq = con.execute(QUERY_TEMPLATE.format(parquet_path)).fetchdf()
elapsed_pq = (time.perf_counter() - t0) / 50 * 1000

# -- CSV -------------------------------------------------------------------
t0 = time.perf_counter()
for _ in range(50):
    result_csv = con.execute(QUERY_TEMPLATE.format(csv_path)).fetchdf()
elapsed_csv = (time.perf_counter() - t0) / 50 * 1000

print(f"Parquet avg query time: {elapsed_pq:.2f} ms")
print(f"CSV     avg query time: {elapsed_csv:.2f} ms")
print(f"Ratio (CSV / Parquet):  {elapsed_csv / elapsed_pq:.1f}x")
print()
print(result_pq)

## Scale up to see the real difference

At 24,000 rows the difference may be small because everything fits in cache.
Let's scale up to 500,000 rows to see the gap widen.

In [ ]:
# ---------------------------------------------------------------------------
# Build a 500K-row version and repeat the benchmark
# ---------------------------------------------------------------------------

N_BIG = 500_000
np.random.seed(99)

df_big = pd.DataFrame({
    "parcel_id": np.arange(N_BIG),
    "observation_date": np.random.choice(DATES, N_BIG),
    "vv_change_db": np.random.normal(-2.0, 3.0, N_BIG),
    "vh_change_db": np.random.normal(-1.5, 2.8, N_BIG),
    "dnbr": np.random.normal(0.15, 0.25, N_BIG),
    "swir_b7": np.random.uniform(0.05, 0.45, N_BIG),
    "ndvi": np.random.normal(0.35, 0.2, N_BIG),
    "parcel_area_m2": np.random.uniform(200, 5000, N_BIG),
    "officially_destroyed": np.random.choice([0, 1], N_BIG, p=[0.7, 0.3]),
})

big_pq = out_dir / "zonal_stats_big.parquet"
big_csv = out_dir / "zonal_stats_big.csv"
df_big.to_parquet(big_pq, index=False)
df_big.to_csv(big_csv, index=False)

print(f"Big Parquet: {big_pq.stat().st_size / 1024 / 1024:.1f} MB")
print(f"Big CSV:     {big_csv.stat().st_size / 1024 / 1024:.1f} MB")

# -- Benchmark -------------------------------------------------------------
N_ITER = 20

t0 = time.perf_counter()
for _ in range(N_ITER):
    con.execute(QUERY_TEMPLATE.format(big_pq)).fetchdf()
elapsed_big_pq = (time.perf_counter() - t0) / N_ITER * 1000

t0 = time.perf_counter()
for _ in range(N_ITER):
    con.execute(QUERY_TEMPLATE.format(big_csv)).fetchdf()
elapsed_big_csv = (time.perf_counter() - t0) / N_ITER * 1000

print(f"\n500K rows -- Parquet: {elapsed_big_pq:.2f} ms")
print(f"500K rows -- CSV:     {elapsed_big_csv:.2f} ms")
print(f"Ratio (CSV / Parquet): {elapsed_big_csv / elapsed_big_pq:.1f}x")

## Verify dbt can read Parquet

dbt-duckdb uses DuckDB under the hood and can query Parquet files as external
sources. Here we confirm that our file is well-formed by reading it back and
inspecting the schema.

In [ ]:
# ---------------------------------------------------------------------------
# Read the parquet back through DuckDB and inspect the schema
# ---------------------------------------------------------------------------

schema = con.execute(f"DESCRIBE SELECT * FROM '{parquet_path}'").fetchdf()
print("Schema of zonal_stats_synthetic.parquet:\n")
print(schema.to_string(index=False))

row_count = con.execute(f"SELECT COUNT(*) FROM '{parquet_path}'").fetchone()[0]
print(f"\nRow count: {row_count:,}")

# Quick sanity query -- same style a dbt model would use
sample = con.execute(f"""
    SELECT observation_date,
           ROUND(AVG(vv_change_db), 3) AS mean_vv,
           ROUND(AVG(ndvi), 3)         AS mean_ndvi,
           SUM(officially_destroyed)    AS n_destroyed
    FROM   '{parquet_path}'
    GROUP  BY observation_date
    ORDER  BY observation_date
""").fetchdf()
print("\nSample aggregation:\n")
print(sample.to_string(index=False))

con.close()
print("\nDone -- Parquet file is valid and queryable.")